# Step 0 · SBERT scoring (Colab)

Run this notebook on Colab with a GPU when available. Upload the run's `df_sbert_input.parquet` and `df_sbert_input.parquet.receipt.json` together when prompted. The receipt supplies the model and positive-only description chosen in data preparation. Download both returned files and stage them together in the same run's `colab_return/` directory on TIP.


In [ ]:
%pip install -q sentence-transformers pyarrow

from importlib.metadata import version
import hashlib
import json
from pathlib import Path
import tempfile

def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


In [ ]:
from google.colab import files

transfer_dir = Path(tempfile.mkdtemp(prefix="patent-sbert-"))
files.upload(target_dir=str(transfer_dir))
input_file = transfer_dir / "df_sbert_input.parquet"
receipt_file = transfer_dir / "df_sbert_input.parquet.receipt.json"
receipt = json.loads(receipt_file.read_text("utf-8"))
input_receipt_hash = file_sha256(receipt_file)
if receipt["path"] != "preparation/df_sbert_input.parquet" or file_sha256(input_file) != receipt["sha256"]:
    raise ValueError("The uploaded Parquet must match the run's input receipt.")
settings = receipt["metadata"]["methodology"]
SBERT_MODEL = settings["sbert_model"]
SBERT_DESCRIPTION = settings["sbert_description"]
description_hash = hashlib.sha256(SBERT_DESCRIPTION.encode("utf-8")).hexdigest()
if description_hash != settings["sbert_description_sha256"]:
    raise ValueError("The SBERT description differs from its recorded fingerprint.")


In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

df = pd.read_parquet(input_file)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Applications:", len(df), "| device:", device)


In [ ]:
# Text of each application: title + abstract.
texts = df["appln_title"].fillna("") + ". " + df["appln_abstract"].fillna("")
unique_texts = texts.drop_duplicates().tolist()
print("Distinct texts to encode:", len(unique_texts))

model = SentenceTransformer(SBERT_MODEL, device=device)

embeddings = model.encode(
    unique_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
description_embedding = model.encode(
    SBERT_DESCRIPTION,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Cosine similarity (the vectors are normalised), then back to one row per application.
score_by_text = pd.Series(embeddings @ description_embedding, index=unique_texts)
df["sbert_similarity"] = texts.map(score_by_text).astype("float32").to_numpy()

assert df["sbert_similarity"].notna().all()
df["sbert_similarity"].describe()

In [ ]:
scored_file = transfer_dir / "df_sbert_scored.parquet"
handoff_file = transfer_dir / "df_sbert_scored.parquet.handoff.json"
if scored_file.exists() or handoff_file.exists():
    raise FileExistsError("Existing scoring output. Re-download it, or start a fresh transfer.")
df.to_parquet(scored_file, index=False)
handoff = {
    "env": receipt["env"], "run_id": receipt["run_id"], "config_sha256": receipt["config_sha256"],
    "input_sha256": receipt["sha256"], "input_receipt_sha256": input_receipt_hash,
    "scored_sha256": file_sha256(scored_file),
    "sbert_model": SBERT_MODEL, "sbert_description_sha256": description_hash,
    "model_revision": getattr(getattr(getattr(model[0], "auto_model", None), "config", None), "_commit_hash", None),
    "device": device, "batch_size": 128, "sentence_transformers_version": version("sentence-transformers"),
    "torch_version": torch.__version__,
}
handoff_file.write_text(json.dumps(handoff, indent=2) + "\n", encoding="utf-8")
files.download(str(scored_file))
files.download(str(handoff_file))
